<br>
    <h1>
        DeepFake Detection Testing
    </h1>
<br>

In [3]:
# Importing required Libraries

import numpy as np
from mtcnn import MTCNN
from PIL import Image
import tensorflow as tf

In [4]:
# Function to detect and crop face

def detect_and_crop_face(image_path):
    # Load the image
    image = Image.open(image_path)
    image = image.convert('RGB')  # Ensure the image is in RGB mode
    image_np = np.array(image)

    # Initialize MTCNN detector
    detector = MTCNN()

    # Detect faces in the image
    detections = detector.detect_faces(image_np)

    if not detections:
        raise ValueError("No face detected in the image.")

    # Assume the first detected face is the target
    x, y, width, height = detections[0]['box']
    x, y = max(0, x), max(0, y)  # Ensure non-negative
    cropped_face = image.crop((x, y, x + width, y + height))

    return cropped_face

In [5]:
# Function to preprocess the cropped face
def preprocess_image(image, target_size=(224, 224)):
    image = image.resize(target_size)  # Resize to the model's expected input size
    image_array = np.array(image) / 255.0  # Normalize pixel values to [0, 1]
    return np.expand_dims(image_array, axis=0)  # Add batch dimension


In [6]:
# Function to classify the face as real or fake
def classify_face(model_path, image_path):
    # Detect and crop the face
    cropped_face = detect_and_crop_face(image_path)
    cropped_face.show()  # Optional: Display the cropped face

    # Preprocess the cropped face
    preprocessed_face = preprocess_image(cropped_face)

    # Load the trained model
    model = tf.keras.models.load_model(model_path)

    # Predict
    prediction = model.predict(preprocessed_face)
    label = "Fake" if prediction[0][0] >= 0.5 else "Real"

    return label, prediction[0][0]

In [16]:
image_path = r"C:\Users\manishpra\OneDrive\Desktop\College things\DAIICT\Semester-1\Foundational Machine Learning\Project\DeepFake Project Implementation\data\Individual pic\Dhruv_2.jpg" 
model_path = r"deepfake-xception-2-trainable.keras"  # Replace with your trained model path

try:
    label, confidence = classify_face(model_path, image_path)
    print(f"Prediction: {label} (Confidence: {confidence:.2f})")
except Exception as e:
    print(f"Error: {e}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
Prediction: Real (Confidence: 0.00)
